# Adversarial RL-Driven Intrusion Detection System (ARL-IDS)

## Abstract
This notebook implements a **Competitive Multi-Agent Reinforcement Learning (MARL)** framework for IOT Intrusion Detection. It features:
1.  **Defender Agent**: A Double DQN (DDQN) based IDS.
2.  **Attacker Agent**: A reinforcement learning adversary that learns to evade detection.
3.  **Joint Evaluation**: A rigorous protocol to measure the robustness gap and attack success rate (ASR).

### Recent Improvements (Paranoid Defender Fix)
We have capped the dynamic reward shaping weights to max **3.0** (previously 10.0). This prevents the agent from becoming "Paranoid" (classifying all normal traffic as attacks to avoid high penalties).

### Architecture
The system uses a 4-layer architecture:
-   **Layer 1**: Data Preprocessing (MinMax Scaling)
-   **Layer 2**: Representation Learning (Autoencoder)
-   **Layer 3**: Competitive Environment (Zero-Sum Game)
-   **Layer 4**: Agents (Attacker & Defender)

---

## 1. Setup Environment
Clone the repository and install necessary dependencies.

In [ ]:
# Clone the Repository
!git clone https://github.com/khalil0401/Adversarial-RL-driven-Intrusion-Detection-System-ARL-IDS-.git
%cd Adversarial-RL-driven-Intrusion-Detection-System-ARL-IDS-

# Install Dependencies
!pip install numpy pandas torch scikit-learn gymnasium matplotlib

## 2. Configuration
Define the hyperparameters for the competitive training loop. 
> **Note**: We set `episodes = 80000` which is sufficient for convergence in the zero-sum game.

In [ ]:
import sys
import os
import argparse

# Add source to python path
sys.path.append(os.getcwd())

from src.train import train
from src.evaluate import evaluate
from src.visualization.plot_results import parse_logs, plot_training

class Config:
    def __init__(self):
        # Dataset Parameters
        # On Kaggle, this is likely "/kaggle/input/ton-iot-network-dataset/train_test_network.csv"
        self.data_path = "/kaggle/input/ton-iot-network-dataset/train_test_network.csv"
        
        # Training Hyperparameters
        self.seed = 42
        self.episodes = 80000        # Recommended: 80,000 for competitive convergence
        self.encoder_epochs = 100     # Pre-training epochs for Autoencoder
        self.lr = 1e-3                # Learning Rate
        self.epsilon_decay = 0.99995  # Slow decay for long exploration
        self.target_update_freq = 1000
        self.weight_update_freq = 1000
        
        # System Flags
        self.skip_encoder_train = False
        self.max_steps_per_episode = 1
        self.no_reward_shaping = False # Enable F1-based Dynamic Reward Shaping
        self.no_adversary = False      # Enable RL Attacker
        self.no_curriculum = False
        
        # Evaluation Paths
        self.defender_path = "results/checkpoints/policy_net.pth"
        self.attacker_path = "results/checkpoints/attacker_net.pth"

args = Config()

print(f"Configuration Loaded. Target Episodes: {args.episodes}")

## 3. Competitive Training Pipeline
The following cell runs the `train()` function which orchestrates:
1.  **Data Loading**: Mins-Max normalization of ToN_IoT data.
2.  **Representation Learning**: Training the Autoencoder.
3.  **Competitive Loop**: Alternating turns between Attacker (Perturb) and Defender (Detect).
4.  **Logging**: Tracks scores and epsilon decay.

In [ ]:
# Redirect stdout to a file to capture logs for visualization
import io
from contextlib import redirect_stdout

log_capture_string = io.StringIO()

if __name__ == "__main__":
    if not os.path.exists(args.data_path):
        print(f"[WARNING] Dataset not found at {args.data_path}.")
        print("Creating Dummy Data for Demonstration...")
    
    print("Starting Competitive Training...")
    
    # We need to capture the logger output specifically, or just ensure logging prints to stdout
    # Creating a file handler for the root logger to save 'training.log'
    import logging
    logging.basicConfig(level=logging.INFO)
    root_logger = logging.getLogger()
    file_handler = logging.FileHandler('training.log', mode='w')
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
    root_logger.addHandler(file_handler)
    
    train(args)
    
    print("Training Complete. Logs saved to training.log")

## 4. Visualization & Analysis
We parse the `training.log` to generate plots showing the convergence of the zero-sum game.

In [ ]:
if os.path.exists('training.log'):
    print("Parsing Logs and Generating Plots...")
    data = parse_logs('training.log')
    plot_training(data)
    
    # Display Plots
    from IPython.display import Image, display
    print("--- Competitive Scores (Defender vs Attacker) ---")
    display(Image(filename='results/plots/competitive_scores.png'))
    
    print("\n--- Class-wise F1 Score History ---")
    display(Image(filename='results/plots/class_f1_history.png'))
else:
    print("Log file not found. Run training first.")

## 5. Joint Evaluation (Robustness Assessment)
After training, we evaluate both agents using the **Joint Policy Assessment** protocol:
-   **Clean Pass**: Standard IDS accuracy.
-   **Adversarial Pass**: IDS accuracy against the trained RL Attacker.
-   **Metrics**: Robustness Gap, Attack Success Rate (ASR), and Confidence Drop.

In [ ]:
if __name__ == "__main__":
    print("Starting Joint Evaluation...")
    evaluate(args)

### View Detailed Results
Display the generated detailed report.

In [ ]:
result_file = "results/joint_evaluation_results.txt"
if os.path.exists(result_file):
    with open(result_file, "r") as f:
        print(f.read())
else:
    print("Results file not found. ensure evaluation ran successfully.")